# TF-IDF Document Similarity Checker

A step-by-step build of a text similarity checker using Python and NumPy only.

**Pipeline:** preprocessing, tokenization, stopword removal, stemming, Bag of Words, TF-IDF, cosine similarity.

**Final result:** `similarity(text_a, text_b)` returns a score from 0 to 1.

Run the cells from top to bottom.

In [20]:
import re
import math
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.4.6


In [21]:
DOCUMENTS = [
    "Machine learning helps computers learn from data and make predictions.",
    "Computers use machine learning to learn patterns from data and predict outcomes.",
    "Deep learning uses neural networks to recognize images and speech.",
    "A healthy vegetarian dinner includes beans, vegetables, and whole grains.",
    "Easy pasta recipes with tomato sauce make a quick dinner.",
    "Marathon runners train for months to improve their stamina.",
    "Cheap flights and hotels make a weekend trip affordable.",
    "Data analysts clean data and build charts to find patterns.",
]
print("Number of documents:", len(DOCUMENTS))

Number of documents: 8


In [22]:
def preprocess(text):
    text = text.lower()                          # 1. lowercase
    text = re.sub(r"[^a-z0-9\s]", " ", text)     # 2. replace non letters/digits/spaces
    return text.split()                          # 3. split into words

sample = DOCUMENTS[0]
print("Original :", sample)
print("Lowercase:", sample.lower())
print("No punct.:", re.sub(r"[^a-z0-9\s]", " ", sample.lower()))
print("Tokens   :", preprocess(sample))

Original : Machine learning helps computers learn from data and make predictions.
Lowercase: machine learning helps computers learn from data and make predictions.
No punct.: machine learning helps computers learn from data and make predictions 
Tokens   : ['machine', 'learning', 'helps', 'computers', 'learn', 'from', 'data', 'and', 'make', 'predictions']


In [23]:
tokenized = [preprocess(doc) for doc in DOCUMENTS]

for i, tokens in enumerate(tokenized):
    print(f"Doc {i}: {len(tokens)} tokens")

shared = set(tokenized[0]) & set(tokenized[1])
print("\nWords shared by doc 0 and doc 1:", sorted(shared))
print("Only in doc 0:", sorted(set(tokenized[0]) - set(tokenized[1])))
print("Only in doc 1:", sorted(set(tokenized[1]) - set(tokenized[0])))

Doc 0: 10 tokens
Doc 1: 12 tokens
Doc 2: 10 tokens
Doc 3: 10 tokens
Doc 4: 10 tokens
Doc 5: 9 tokens
Doc 6: 9 tokens
Doc 7: 10 tokens

Words shared by doc 0 and doc 1: ['and', 'computers', 'data', 'from', 'learn', 'learning', 'machine']
Only in doc 0: ['helps', 'make', 'predictions']
Only in doc 1: ['outcomes', 'patterns', 'predict', 'to', 'use']


In [24]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "have", "in", "into", "is", "it", "its", "of", "on", "or",
    "that", "the", "their", "to", "was", "were", "will", "with",
}

tokens = preprocess(DOCUMENTS[0])
filtered = [t for t in tokens if t not in STOPWORDS]
print("Before:", tokens)
print("After :", filtered)
print("Removed:", [t for t in tokens if t in STOPWORDS])

Before: ['machine', 'learning', 'helps', 'computers', 'learn', 'from', 'data', 'and', 'make', 'predictions']
After : ['machine', 'learning', 'helps', 'computers', 'learn', 'data', 'make', 'predictions']
Removed: ['from', 'and']


In [25]:
def simple_stem(word):
    # remove the first matching ending, but keep at least 3 letters
    for suffix in ["ions", "ion", "ing", "ers", "ed", "es", "s"]:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)]
    return word

In [26]:
from nltk.stem import PorterStemmer
porter = PorterStemmer()

words = ["learning", "predictions", "computers", "outcomes", "recipes", "healthy", "vegetables"]
print(f"{'word':12} {'ours':10} {'porter':10}")
for w in words:
    print(f"{w:12} {simple_stem(w):10} {porter.stem(w):10}")

word         ours       porter    
learning     learn      learn     
predictions  predict    predict   
computers    comput     comput    
outcomes     outcom     outcom    
recipes      recip      recip     
healthy      healthy    healthi   
vegetables   vegetabl   veget     


In [27]:
def full_preprocess(text):
    tokens = preprocess(text)                           # Step 2
    tokens = [t for t in tokens if t not in STOPWORDS]  # remove filler
    return [simple_stem(t) for t in tokens]             # reduce to roots

processed = [full_preprocess(doc) for doc in DOCUMENTS]

print("Doc 0:", processed[0])
print("Doc 1:", processed[1])
print("\nShared stems     :", sorted(set(processed[0]) & set(processed[1])))
print("Only in doc 0    :", sorted(set(processed[0]) - set(processed[1])))
print("Only in doc 1    :", sorted(set(processed[1]) - set(processed[0])))

Doc 0: ['machine', 'learn', 'help', 'comput', 'learn', 'data', 'make', 'predict']
Doc 1: ['comput', 'use', 'machine', 'learn', 'learn', 'pattern', 'data', 'predict', 'outcom']

Shared stems     : ['comput', 'data', 'learn', 'machine', 'predict']
Only in doc 0    : ['help', 'make']
Only in doc 1    : ['outcom', 'pattern', 'use']


In [28]:
vocab = sorted(set(token for doc in processed for token in doc))
word_to_idx = {word: i for i, word in enumerate(vocab)}

print("Vocabulary size:", len(vocab))
print(vocab)

Vocabulary size: 47
['affordable', 'analyst', 'bean', 'build', 'chart', 'cheap', 'clean', 'comput', 'data', 'deep', 'dinner', 'easy', 'find', 'flight', 'grain', 'healthy', 'help', 'hotel', 'imag', 'improve', 'includ', 'learn', 'machine', 'make', 'marathon', 'month', 'network', 'neural', 'outcom', 'pasta', 'pattern', 'predict', 'quick', 'recip', 'recognize', 'runn', 'sauce', 'speech', 'stamina', 'tomato', 'train', 'trip', 'use', 'vegetabl', 'vegetarian', 'weekend', 'whole']


In [29]:
# STEP 4: BAG OF WORDS - turn each document into a row of word counts

def bow_vector(tokens):
    vec = np.zeros(len(vocab))              # one slot per vocabulary word, all 0
    for token in tokens:
        if token in word_to_idx:            # skip unknown words (avoids KeyError)
            vec[word_to_idx[token]] += 1    # add 1 at this word's position
    return vec                              # word counts for one document

# one row per document, one column per word, e.g. (8, 47)
bow = np.array([bow_vector(doc) for doc in processed])
print("BoW matrix shape:", bow.shape, "(documents x words)")

# check: print the words that appear in doc 0, with their counts
print("\nDoc 0 :", processed[0])
for i in np.nonzero(bow[0])[0]:             # positions where the count is not 0
    print(f"   {vocab[i]:10} {int(bow[0][i])}")

BoW matrix shape: (8, 47) (documents x words)

Doc 0 : ['machine', 'learn', 'help', 'comput', 'learn', 'data', 'make', 'predict']
   comput     1
   data       1
   help       1
   learn      2
   machine    1
   make       1
   predict    1


In [30]:
# raw dot product between documents (the dot product idea you already know)
print("doc0 . doc1 (same topic, ML)      :", bow[0] @ bow[1])
print("doc0 . doc4 (pasta recipe)        :", bow[0] @ bow[4])
print("doc0 . doc7 (data analysis)       :", bow[0] @ bow[7])

# which words do doc 0 and doc 4 share?
shared = [vocab[i] for i in range(len(vocab)) if bow[0][i] > 0 and bow[4][i] > 0]
print("\nWords shared by doc 0 and doc 4:", shared)

doc0 . doc1 (same topic, ML)      : 8.0
doc0 . doc4 (pasta recipe)        : 1.0
doc0 . doc7 (data analysis)       : 2.0

Words shared by doc 0 and doc 4: ['make']


In [31]:
# IDF: how rare is each word across all documents?
N = len(DOCUMENTS)                  # number of documents
df = (bow > 0).sum(axis=0)          # in how many documents each word appears
idf = np.log(N / df)                # rare word -> big idf, common word -> small idf

order = np.argsort(idf)             # word positions from lowest idf to highest

print("Most common words (lowest idf):")
for i in order[:5]:
    print(f"  {vocab[i]:10} in {df[i]} of {N} docs   idf = {idf[i]:.3f}")

print("\nRarest words (highest idf):")
for i in order[-5:]:
    print(f"  {vocab[i]:10} in {df[i]} of {N} docs   idf = {idf[i]:.3f}")

Most common words (lowest idf):
  data       in 3 of 8 docs   idf = 0.981
  make       in 3 of 8 docs   idf = 0.981
  learn      in 3 of 8 docs   idf = 0.981
  comput     in 2 of 8 docs   idf = 1.386
  pattern    in 2 of 8 docs   idf = 1.386

Rarest words (highest idf):
  trip       in 1 of 8 docs   idf = 2.079
  vegetabl   in 1 of 8 docs   idf = 2.079
  vegetarian in 1 of 8 docs   idf = 2.079
  weekend    in 1 of 8 docs   idf = 2.079
  whole      in 1 of 8 docs   idf = 2.079


In [32]:
# IDF
N = len(DOCUMENTS)
df = (bow > 0).sum(axis=0)
idf = np.log(N / df)

# TF-IDF
tf = bow / bow.sum(axis=1, keepdims=True)
tfidf = tf * idf
print("TF-IDF matrix shape:", tfidf.shape)

TF-IDF matrix shape: (8, 47)


In [33]:
print("                         raw counts    tf-idf")
for name, j in [("doc0 . doc1 (ML pair)", 1),
                ("doc0 . doc4 (pasta)   ", 4),
                ("doc0 . doc7 (data)    ", 7)]:
    print(f"{name}   {bow[0] @ bow[j]:8.1f}   {tfidf[0] @ tfidf[j]:8.4f}")

                         raw counts    tf-idf
doc0 . doc1 (ML pair)        8.0     0.1469
doc0 . doc4 (pasta)           1.0     0.0150
doc0 . doc7 (data)            2.0     0.0301


In [34]:
def cosine_similarity(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:      # a text with no known words
        return 0.0
    return float(a @ b / (norm_a * norm_b))

print("length of doc 0 vector:", round(float(np.linalg.norm(tfidf[0])), 4))
print("length of doc 1 vector:", round(float(np.linalg.norm(tfidf[1])), 4))

print("\n                         dot product   cosine")
for name, j in [("doc0 vs doc1 (ML pair)", 1),
                ("doc0 vs doc4 (pasta)   ", 4),
                ("doc0 vs doc7 (data)    ", 7)]:
    print(f"{name}   {tfidf[0] @ tfidf[j]:10.4f}   {cosine_similarity(tfidf[0], tfidf[j]):.4f}")

length of doc 0 vector: 0.4978
length of doc 1 vector: 0.481

                         dot product   cosine
doc0 vs doc1 (ML pair)       0.1469   0.6133
doc0 vs doc4 (pasta)          0.0150   0.0450
doc0 vs doc7 (data)           0.0301   0.0923


In [35]:
scores = [(j, cosine_similarity(tfidf[0], tfidf[j])) for j in range(len(DOCUMENTS))]
scores.sort(key=lambda pair: pair[1], reverse=True)

print("Documents ranked by similarity to doc 0:")
for j, s in scores:
    print(f"  {s:.3f}  doc {j}: {DOCUMENTS[j][:60]}")

Documents ranked by similarity to doc 0:
  1.000  doc 0: Machine learning helps computers learn from data and make pr
  0.613  doc 1: Computers use machine learning to learn patterns from data a
  0.092  doc 7: Data analysts clean data and build charts to find patterns.
  0.090  doc 2: Deep learning uses neural networks to recognize images and s
  0.047  doc 6: Cheap flights and hotels make a weekend trip affordable.
  0.045  doc 4: Easy pasta recipes with tomato sauce make a quick dinner.
  0.000  doc 3: A healthy vegetarian dinner includes beans, vegetables, and 
  0.000  doc 5: Marathon runners train for months to improve their stamina.


In [36]:
def build_tfidf(texts):
    token_lists = [full_preprocess(t) for t in texts]
    vocab_ = sorted(set(tok for toks in token_lists for tok in toks))
    idx = {w: i for i, w in enumerate(vocab_)}

    counts = np.zeros((len(texts), len(vocab_)))
    for r, toks in enumerate(token_lists):
        for tok in toks:
            counts[r, idx[tok]] += 1

    df_ = (counts > 0).sum(axis=0)
    idf_ = np.log(len(texts) / df_)

    lengths = counts.sum(axis=1, keepdims=True)
    lengths[lengths == 0] = 1            # an empty text stays all zeros
    tf_ = counts / lengths
    return tf_ * idf_

def similarity(text_a, text_b, background=DOCUMENTS):
    texts = list(background) + [text_a, text_b]
    matrix = build_tfidf(texts)
    return cosine_similarity(matrix[-2], matrix[-1])    # the last two rows

In [37]:
a = "Machine learning helps computers learn from data and make predictions."
b = "Computers use machine learning to learn patterns from data and predict outcomes."
c = "Easy pasta recipes with tomato sauce make a quick dinner."

print("ML pair             :", round(similarity(a, b), 3))
print("ML vs pasta         :", round(similarity(a, c), 3))
print("text with itself    :", round(similarity(a, a), 3))
print("text vs empty text  :", round(similarity(a, ""), 3))

ML pair             : 0.528
ML vs pasta         : 0.034
text with itself    : 1.0
text vs empty text  : 0.0


In [38]:
print("with background    :", round(similarity(a, b), 3))
print("without background :", round(similarity(a, b, background=[]), 3))

with background    : 0.528
without background : 0.0
